# Tau2-Bench: Online Env vs Trace Dataset — Sample-by-Sample Comparison

**Goal**: Compare the tau2-bench online environment (`src/act_prm/environments/tau2bench/`)
with the reasoning trace dataset (`mzio/aprm-amityco_apigen_tau_bench_split_turn`) at the
sample level, and determine whether trace splits can be aligned to the online env's splits.

## Online Env (tau2-bench verified)
- Source: `tau2.run.get_tasks()` loading from `split_tasks.json`
- Airline: 50 tasks (30 train / 20 test), Retail: 114 tasks (74 train / 40 test)
- Task IDs are non-contiguous within splits
- Each task defines: `known_info` (user identity), `reason_for_call`, `task_instructions`
- Requires `TAU2_DATA_DIR=/scr/mzhang/projects/rlic-cc/data/tau2-bench-verified/data`

## Trace Dataset
- Source: `amityco/apigen-tau-bench-split-turn` → processed via `tau_bench.py`
- Airline: 1,586 UIDs (11,683 rows), Retail: 3,410 UIDs (33,583 rows)
- All `generation_id=0`
- `unique_data_sample_id` is a sequential counter, NOT a tau2 task ID

In [ ]:
import json
import re
from datasets import load_dataset
from collections import Counter

## 1. Tau2 Online Env: Task Splits & Sample Structure

```
airline split=train: 30 tasks (IDs: 0,1,3,4,5,7,9,10,11,12,14,15,17,20,21,23,27,28,33,34,36,38,39,40,41,42,43,46,47,49)
airline split=test:  20 tasks (IDs: 2,6,8,13,16,18,19,22,24,25,26,29,30,31,32,35,37,44,45,48)

retail split=train: 74 tasks
retail split=test:  40 tasks
```

Each tau2 task looks like (airline task 0 example):
```json
{
  "id": "0",
  "known_info": "You are Emma Kim.\nYour user id is emma_kim_9957.",
  "reason_for_call": "You want to cancel reservation EHGLP3.\nIt may be more than 24 hours after booking...",
  "task_instructions": "If Agent tells you that cancellation is not possible, mention that..."
}
```

## 2. Trace Dataset: Sample Structure

In [ ]:
traces = load_dataset('mzio/aprm-amityco_apigen_tau_bench_split_turn')

print('Trace dataset splits:')
for name, ds in traces.items():
    uids = sorted(set(ds['unique_data_sample_id']))
    gen_ids = sorted(set(ds['generation_id']))
    print(f'  {name}: {len(ds)} rows, {len(uids)} UIDs, '
          f'uid range [{min(uids)}, {max(uids)}], gen_ids={gen_ids}')

# Show sample trace structure
print('\n--- Sample trace (airline, UID 0, timestep 0) ---')
ds = traces['airline']
row = [ds[i] for i in range(len(ds)) if ds[i]['unique_data_sample_id'] == 0 and ds[i]['timestep'] == 0][0]
print(f"  Columns: {list(row.keys())}")
print(f"  domain: {row['domain']}")
print(f"  system_prompt (first 150): {row['system_prompt'][:150]}...")
print(f"  state ({len(row['state'])} msgs):")
for m in row['state']:
    print(f"    [{m['role']}]: {m['content'][:150]}")
print(f"  action: {row['action']['content'][:150]}...")
print(f"  reward={row['reward']}, return_={row['return_']}, done={row['done']}")

## 3. Side-by-Side: Tau2 Tasks vs Trace Samples

Let's compare the first few tau2 tasks with the first few trace UIDs to see the differences.

In [ ]:
# tau2_task_full_info.json was generated on hazy1 via:
#   export TAU2_DATA_DIR=/scr/mzhang/projects/rlic-cc/data/tau2-bench-verified/data
#   python -c "... from tau2.utils.io_utils import load_file ..."
# Contains {domain: {task_id: {known_info, reason_for_call, task_instructions}}}
with open('/tmp/tau2_task_full_info.json') as f:
    tau2_tasks = json.load(f)

ds = traces['airline']

# Get first user message and user_id from tool results for each trace UID
uid_details = {}
for i in range(len(ds)):
    row = ds[i]
    uid = row['unique_data_sample_id']
    if uid not in uid_details:
        uid_details[uid] = {'first_msg': '', 'user_ids': set(), 'reservation_ids': set()}
    for msg in row['state']:
        content = msg.get('content', '') or ''
        for m in re.finditer(r'"user_id":\s*"([^"]+)"', content):
            uid_details[uid]['user_ids'].add(m.group(1))
        for m in re.finditer(r'"reservation_id":\s*"([^"]+)"', content):
            uid_details[uid]['reservation_ids'].add(m.group(1))
    action = row['action']
    if isinstance(action, dict):
        content = action.get('content', '') or ''
        for m in re.finditer(r'"user_id":\s*"([^"]+)"', content):
            uid_details[uid]['user_ids'].add(m.group(1))
        for m in re.finditer(r'"reservation_id":\s*"([^"]+)"', content):
            uid_details[uid]['reservation_ids'].add(m.group(1))
    if row['timestep'] == 0:
        for msg in row['state']:
            if msg['role'] == 'user':
                uid_details[uid]['first_msg'] = msg['content']
                break

# Side-by-side comparison
print("=" * 80)
print("SIDE-BY-SIDE: Tau2 Tasks vs Trace UIDs (airline)")
print("=" * 80)

for idx in range(5):
    task_id = str(idx)
    info = tau2_tasks['airline'][task_id]
    uid_info = uid_details.get(idx, {})

    print(f"\n{'─' * 80}")
    print(f"  TAU2 TASK {task_id}:")
    print(f"    user:   {info['known_info'].split(chr(10))[0]}")
    uid_line = [line for line in info['known_info'].split('\n') if 'user' in line.lower() and 'id' in line.lower()]
    if uid_line:
        print(f"    user_id: {uid_line[0].strip()}")
    print(f"    reason: {info['reason_for_call'][:120]}")
    print()
    print(f"  TRACE UID {idx}:")
    print(f"    user_ids:        {uid_info.get('user_ids', set())}")
    print(f"    reservation_ids: {uid_info.get('reservation_ids', set())}")
    print(f"    first_msg:       {uid_info.get('first_msg', '')[:120]}")
    print()
    # Highlight: same user? same reservation?
    task_user_match = re.search(r'(?:user.?id[:\s]+|is\s+)([a-z][a-z_]+\d+)', info['known_info'], re.IGNORECASE)
    task_user_id = task_user_match.group(1) if task_user_match else None
    trace_user_ids = uid_info.get('user_ids', set())
    user_match = task_user_id in trace_user_ids if task_user_id else False
    print(f"    MATCH? user_id={user_match} (tau2: {task_user_id}, trace: {trace_user_ids})")

## 4. Attempting to Map Traces → Tau2 Tasks

Since the traces contain actual tool call results with `user_id` and `reservation_id`,
we can try to match traces to tau2 tasks by comparing these identifiers.

**Strategy**:
1. Extract `user_id` from trace tool results (e.g., `get_reservation_details` responses)
2. Extract `user_id` from tau2 task `known_info`
3. Match by `user_id` + `reservation_id` (strongest), then `user_id` only (weaker)

In [ ]:
## Airline matching

# Extract user_ids from tau2 tasks
task_user_ids = {}
task_res_ids = {}
for task_id, info in tau2_tasks['airline'].items():
    known = info['known_info']
    reason = info['reason_for_call']
    uid_match = re.search(r'(?:user.?id[:\s]+|is\s+)([a-z][a-z_]+\d+)', known, re.IGNORECASE)
    if uid_match:
        task_user_ids[task_id] = uid_match.group(1)
    res_matches = re.findall(r'\b([A-Z][A-Z0-9]{4,7})\b', known + ' ' + reason)
    task_res_ids[task_id] = set(res_matches)

# Check overlap between trace user_ids and tau2 user_ids
trace_all_user_ids = set()
for uid, info in uid_details.items():
    trace_all_user_ids.update(info['user_ids'])
tau2_all_user_ids = set(task_user_ids.values())
overlap = trace_all_user_ids & tau2_all_user_ids

print(f"Unique user_ids in traces: {len(trace_all_user_ids)}")
print(f"Unique user_ids in tau2 tasks: {len(tau2_all_user_ids)}")
print(f"Overlapping user_ids: {len(overlap)}")
print(f"  -> {sorted(overlap)[:10]}...")

# Match: user_id + reservation_id
uid_to_task_strong = {}
for uid, info in uid_details.items():
    for task_id in tau2_tasks['airline']:
        if task_id not in task_user_ids:
            continue
        if task_user_ids[task_id] in info['user_ids']:
            if task_res_ids[task_id] and task_res_ids[task_id] & info['reservation_ids']:
                uid_to_task_strong[uid] = task_id
                break

# Match: user_id only (unique match)
uid_to_task_weak = {}
for uid, info in uid_details.items():
    if uid in uid_to_task_strong:
        continue
    matches = [tid for tid, tuid in task_user_ids.items() if tuid in info['user_ids']]
    if len(matches) == 1:
        uid_to_task_weak[uid] = matches[0]

uid_to_task_all = {**uid_to_task_strong, **uid_to_task_weak}
matched_tasks = set(uid_to_task_all.values())

print("\n--- Airline Matching Results ---")
print(f"  Strong match (user_id + reservation_id): {len(uid_to_task_strong)} UIDs")
print(f"  Weak match (user_id only, unique):       {len(uid_to_task_weak)} UIDs")
print(f"  Total matched: {len(uid_to_task_all)} / {len(uid_details)} UIDs ({100*len(uid_to_task_all)/len(uid_details):.1f}%)")
print(f"  Covering {len(matched_tasks)} / {len(tau2_tasks['airline'])} tau2 tasks")
print(f"  Unmatched UIDs: {len(uid_details) - len(uid_to_task_all)}")
print(f"  Unmatched tau2 tasks: {sorted(set(tau2_tasks['airline'].keys()) - matched_tasks, key=int)}")

if uid_to_task_all:
    tc = Counter(uid_to_task_all.values())
    print(f"\n  Traces per matched task: min={min(tc.values())}, max={max(tc.values())}, mean={sum(tc.values())/len(tc):.1f}")

## 5. Retail Matching

In [ ]:
## Retail matching — same approach
ds_retail = traces['retail']

# Build retail trace details
retail_uid_details = {}
for i in range(len(ds_retail)):
    row = ds_retail[i]
    uid = row['unique_data_sample_id']
    if uid not in retail_uid_details:
        retail_uid_details[uid] = {'first_msg': '', 'user_ids': set(), 'order_ids': set()}
    for msg in row['state']:
        content = msg.get('content', '') or ''
        for m in re.finditer(r'"user_id":\s*"([^"]+)"', content):
            retail_uid_details[uid]['user_ids'].add(m.group(1))
        for m in re.finditer(r'"order_id":\s*"([^"]+)"', content):
            retail_uid_details[uid]['order_ids'].add(m.group(1))
    action = row['action']
    if isinstance(action, dict):
        content = action.get('content', '') or ''
        for m in re.finditer(r'"user_id":\s*"([^"]+)"', content):
            retail_uid_details[uid]['user_ids'].add(m.group(1))
        for m in re.finditer(r'"order_id":\s*"([^"]+)"', content):
            retail_uid_details[uid]['order_ids'].add(m.group(1))
    if row['timestep'] == 0:
        for msg in row['state']:
            if msg['role'] == 'user':
                retail_uid_details[uid]['first_msg'] = msg['content']
                break

# Extract user_ids from tau2 retail tasks
retail_task_user_ids = {}
for task_id, info in tau2_tasks['retail'].items():
    known = info['known_info']
    uid_match = re.search(r'(?:user.?id[:\s]+|is\s+)([a-z][a-z_]+\d+)', known, re.IGNORECASE)
    if uid_match:
        retail_task_user_ids[task_id] = uid_match.group(1)

retail_trace_user_ids = set()
for uid, info in retail_uid_details.items():
    retail_trace_user_ids.update(info['user_ids'])
retail_tau2_user_ids = set(retail_task_user_ids.values())
retail_overlap = retail_trace_user_ids & retail_tau2_user_ids

print("--- Retail Matching Results ---")
print(f"  Tau2 retail tasks with user_id: {len(retail_task_user_ids)} / {len(tau2_tasks['retail'])}")
print(f"  Unique user_ids in traces: {len(retail_trace_user_ids)}")
print(f"  Unique user_ids in tau2: {len(retail_tau2_user_ids)}")
print(f"  Overlapping user_ids: {len(retail_overlap)}")
print("\n  -> Retail tau2 tasks barely specify user_ids, so matching is not possible.")

## 6. System Prompt Comparison

The online env and trace configs use different system prompts. The traces store
the full domain policy in each row's `system_prompt` column, but the act_prm and
act_lm configs override the base prompt.

| Environment | Base System Prompt |
|-------------|-------------------|
| Online (`tau2bench/`) | `"You are a helpful customer service agent."` + `<policy>` block + respond_user instruction |
| Act-PRM traces (`act_prm/taubench_*`) | `"You are a helpful assistant that infers reasoning thoughts behind your own observed actions."` |
| Act-LM traces (`act_lm/taubench_*`) | `"You are a helpful tool-calling assistant."` |

With the updated `build_system_prompt()` (matching TextWorld's pattern), trace
configs can inject their own base prompt while still getting the domain policy appended.

## 7. Conclusion: Trace-to-Task Alignment is Not Feasible

**Key findings:**

1. **Different database states**: The trace dataset (`amityco/apigen-tau-bench-split-turn`) and
   the tau2-bench verified environment use different underlying databases. Reservation codes,
   order IDs, and even some user IDs differ between the two sources.

2. **Airline**: Only ~6.7% of trace UIDs (107/1,586) can be matched to tau2 tasks via user_id,
   covering 27/50 tasks. The remaining 93% of traces have no identifiable tau2 task correspondence.

3. **Retail**: Matching is effectively impossible — only 3/114 tau2 tasks specify user_ids in
   their `known_info`, and none overlap with trace user_ids.

4. **`unique_data_sample_id` ≠ task ID**: The trace dataset's UID is a sequential counter
   assigned during processing (`tau_bench.py` line 210), not a tau2 task ID.

**Implications for train/test splitting:**

- **Online env** (`tau2bench/`): Uses tau2's canonical `split_tasks.json` splits directly.
  Train/test are well-defined and non-overlapping. ✅
- **Trace dataset** (`act_prm/taubench_*`, `act_lm/taubench_*`): Cannot be aligned to the
  online env's splits. The trace `ActionLmEnv` uses its own random split via `frac_train_tasks`
  with `data_seed`, which is independent of tau2's canonical splits.
- **Cross-contamination risk**: Since traces can't be mapped to tau2 tasks, there's no way to
  guarantee that trace-trained models haven't seen test-split tasks. This is acceptable for
  Act-PRM/Act-LM training (where we train on reasoning patterns, not task-specific knowledge),
  but should be noted when reporting results.

**Recommendation**: For tau2bench experiments, treat the online env and trace dataset as
independent data sources with separate split semantics. Do not attempt to align them.